[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_DSP/Graph_Signal_Processing.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Graph Signal Processing & GNNs

A flagship IEEE-SPS research area with almost no accessible teaching material: signals that live on **networks** — sensor grids, social graphs, molecules, power grids. Four sessions: the graph Laplacian gives graphs a Fourier transform, filters, and sampling theory — and message-passing GNNs drop out as learned graph filters. The oracle throughout: on a ring graph, everything must reduce to classical DSP.

## 1. Pre-requisites

- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S3 (eigendecomposition — this course is its victory lap).
- [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) (classical Fourier, for the reduction check).
- [CNN workshop](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) for Session 4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

def laplacian(A):
    return np.diag(A.sum(1)) - A

# our running graph: a random sensor network (geometric graph)
n_nodes = 80
pos = rng.random((n_nodes, 2))
D2 = ((pos[:, None] - pos[None]) ** 2).sum(-1)
A = ((D2 < 0.045) & (D2 > 0)).astype(float)
L = laplacian(A)
lam, U = np.linalg.eigh(L)                          # the graph's "frequencies" and "Fourier basis"

def draw(signal, title="", ax=None):
    if ax is None: fig, ax = plt.subplots(figsize=(3.6, 3.2))
    for i, j in zip(*np.nonzero(np.triu(A))):
        ax.plot(*zip(pos[i], pos[j]), "k-", linewidth=0.3, alpha=0.4)
    sc = ax.scatter(*pos.T, c=signal, s=45, cmap="coolwarm")
    ax.set_title(title, fontsize=9); ax.axis("off")
    return sc

---
### 🕐 Session 1 of 4 — *The Graph Laplacian & Graph Fourier Transform* (~40 min)
**Goal:** give any graph a frequency axis; verify it reduces to the DFT on a ring.
**Builds on:** [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S3. &nbsp; **Feeds into:** Session 2 (filtering on graphs).

---

## 2. Frequency Without Time

💡 **Intuition.** What does 'frequency' mean with no time axis? **Smoothness with respect to the edges.** The Laplacian quadratic form $x^T L x = \sum_{(i,j)\in E}(x_i - x_j)^2$ totals the disagreement across edges — so Laplacian eigenvectors, ordered by eigenvalue, are the graph's own harmonics: $\lambda \approx 0$ ⇒ smooth (neighbors agree), large $\lambda$ ⇒ oscillatory (neighbors alternate). The **graph Fourier transform** is just analysis in this eigenbasis: $\hat{x} = U^T x$ — [Hilbert-space](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) change of basis, with the graph choosing the basis.

In [ ]:

# YOUR CODE HERE


**What just happened.** Four Laplacian eigenvectors of the sensor network, drawn on the network itself and ordered by eigenvalue. Read them left to right and the frequency axis appears.

Eigenvector 0, at $\lambda = 0$, is **constant** — every node the same colour. That is the DC component, and it exists because a constant signal has zero disagreement across every edge, making $x^\top L x = 0$. Every connected graph has exactly one such eigenvector, and a useful diagnostic follows: the *multiplicity* of $\lambda = 0$ counts connected components. Three zero eigenvalues means your "network" is really three networks.

Eigenvector 1 divides the graph into two smoothly-varying regions — the lowest nonzero frequency, and the reason this vector (the **Fiedler vector**) is the basis of spectral clustering: the sign of its entries is a principled two-way partition of the graph.

Eigenvector 4 oscillates more, with several regions. And eigenvector 60, at high $\lambda$, alternates sharply between adjacent nodes — neighbours take opposite colours almost everywhere. That is what high graph frequency *looks like*, and it is the visual analogue of $(-1)^n$ in classical DSP.

**So "frequency" on a graph means smoothness with respect to the edges**, and $\lambda$ measures it precisely. The Laplacian quadratic form $x^\top L x = \sum_{(i,j)\in E}(x_i - x_j)^2$ totals the disagreement across edges, so ordering eigenvectors by eigenvalue orders them from smoothest to most oscillatory. There is no time axis here and no notion of "one sample later" — and none is needed. The graph's connectivity alone defines what "slowly varying" means.

The graph Fourier transform is then just $\hat x = U^\top x$: analysis in this eigenbasis, which is the [Hilbert space](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) change-of-basis idea with the *graph* choosing the basis rather than a mathematician. The next cell checks that when the graph is a ring, the basis it chooses is the one we already know.

In [ ]:
# ORACLE: on a RING graph, the Laplacian eigenvalues must be the classical DFT frequencies

# YOUR CODE HERE


**What just happened.** The ring graph's Laplacian eigenvalues match $2 - 2\cos(2\pi k/N)$ to **3.6e-15** — machine precision. This is the most important cell in the workshop, and it is not a sanity check.

**It is a reduction.** A ring is the graph whose nodes are arranged in a cycle with each connected to its two neighbours — which is exactly the structure of a periodic discrete-time signal. Its Laplacian is diagonalised by the DFT basis, and its eigenvalues are precisely the classical frequency response of the second-difference operator. So **classical DSP is the special case of graph signal processing where the graph is a ring.** The new theory contains the old one rather than competing with it.

That reframing is worth stating in full. Every tool from the DSP track — Fourier analysis, filtering, sampling, convolution — was implicitly assuming a graph all along: a regular, translation-invariant one. Images assume a 2-D grid graph; video adds a temporal edge. GSP simply removes the assumption that the graph is regular, and asks what survives. Most of it does.

**Why $2 - 2\cos(2\pi k/N)$ specifically.** The ring Laplacian applied to a signal computes $2x_n - x_{n+1} - x_{n-1}$, the negative second difference. Feed it the complex exponential $e^{j2\pi kn/N}$ and it comes back scaled by $2 - e^{j2\pi k/N} - e^{-j2\pi k/N} = 2 - 2\cos(2\pi k/N)$. The exponentials are eigenvectors, and those are the eigenvalues — the same "complex exponentials diagonalise shift-invariant operators" fact from [Hilbert Spaces](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb), now visibly a graph statement.

Note the shape of that eigenvalue curve: it starts at 0 for $k=0$ and rises to 4 at $k = N/2$. Low $k$ means low $\lambda$ means smooth, exactly as the general theory claims — and $\lambda$ really is playing the role of $\omega$.

**And the general definition earns its keep here.** $x^\top L x = \sum_{(i,j)\in E}(x_i - x_j)^2$ totals the disagreement across edges, so ordering eigenvectors by $\lambda$ orders them from smoothest to most oscillatory. On the sensor network above, eigenvector 0 is constant (every connected graph has exactly one $\lambda = 0$ eigenvector — and the multiplicity of zero counts connected components, a useful diagnostic), while eigenvector 60 alternates sharply between neighbours. That is what high frequency looks like when there is no time axis.

---
### 🕐 Session 2 of 4 — *Filtering on Graphs* (~40 min)
**Goal:** denoise a sensor field with a graph low-pass; make it local with polynomial filters.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (sampling).

---

## 3. Graph Filters

💡 **Intuition.** A graph filter scales each harmonic: $y = U h(\Lambda) U^T x$ — design $h(\lambda)$ exactly like a [filter response](./Filter_Design.ipynb), with $\lambda$ replacing $\omega$. The practical twist: eigendecomposition is $O(n^3)$, but a **polynomial** filter $h(L) = \sum_k c_k L^k$ needs only matrix-vector products — and $L^k x$ touches only $k$-hop neighbors, so polynomial order = *filter locality*. That locality is the seed GNNs grow from.

In [ ]:
# denoise a smooth temperature field over the sensor network
# local polynomial approximation of the same filter (Chebyshev-lite: least-squares fit)

# YOUR CODE HERE


**What just happened.** Both filters cleaned up the noisy sensor field — **7.6 dB** for the exact spectral filter, **8.6 dB** for the 5-hop polynomial. Denoising on an irregular network, using a frequency axis that only exists because we defined one.

**The design was a one-line translation.** `h = 1.0 / (1.0 + 4.0 * lam)` is a low-pass: near 1 at $\lambda \approx 0$ (smooth harmonics pass), rolling off as $\lambda$ grows (oscillatory harmonics suppressed). That is exactly how you would design a [classical frequency response](./Filter_Design.ipynb), with $\lambda$ substituted for $\omega$. And it works for the same reason as always: the signal was built from the first four harmonics, so it is smooth, while the added noise spreads across *all* harmonics. Low-pass keeps the signal and discards most of the noise.

**But look at the polynomial version, because it is the practically important one.** Computing $U$ requires an $O(n^3)$ eigendecomposition, which is hopeless for a graph with a million nodes and — worse — *global*, since every output depends on every input. The polynomial filter $\sum_k c_k L^k$ needs only matrix–vector products, and it has a much deeper property: **$L^k x$ at node $i$ depends only on nodes within $k$ hops.** Polynomial order *is* filter locality.

That is what makes graph filtering deployable. A 5-hop filter can run on the sensor network itself — each node exchanging values with its immediate neighbours, five times, with no central computer and no node ever knowing the global graph. It is also the seed from which every GNN grows, which Session 4 makes explicit.

**Now the result that looks impossible: the approximation beat the exact filter.** 8.6 dB against 7.6 dB — the polynomial fit outperformed the thing it was fitting. Resolve it carefully rather than moving on.

The spectral filter is *exact* for the response `h`, but `h` itself was chosen by hand: the `4.0` was picked, not optimised. So it is a reasonable low-pass, not the optimal one for this problem, and there is no theorem saying it should win. The degree-5 least-squares fit does not reproduce `h` exactly — its mismatch is presumably a slightly more aggressive roll-off — and on this particular signal and noise realisation that mismatch happens to help.

The honest reading is therefore **not** "polynomials are better." It is that the reference filter was never optimal, and a 1 dB gap measured on a single noise realisation is within run-to-run variation regardless. Re-run with a few different seeds and the ordering will swap. A one-run difference of this size is not a finding, and treating it as one is the exact mistake this workshop's oracle-driven style exists to prevent.

---
### 🕐 Session 3 of 4 — *Sampling on Graphs* (~35 min)
**Goal:** which sensors can you afford to lose? Bandlimited recovery from a subset of nodes.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (GNNs).

---

## 4. Nyquist for Networks

💡 **Intuition.** If a graph signal is **bandlimited** — lives in the span of the first $K$ harmonics — then $K$ well-chosen node readings determine *all* $n$: solve the little least-squares system in the known coefficients ([Compressed Sensing's](./Compressed_Sensing.ipynb) logic, subspace version). 'Well-chosen' matters exactly like array geometry: sample nodes that make the harmonics distinguishable, not clustered clones of each other.

In [ ]:
# and the failure mode: measure fewer than K nodes → underdetermined

# YOUR CODE HERE


**What just happened.** All **80** node values reconstructed from **12** sensor readings, to a maximum error of **2.2e-15** — machine precision. And then the failure: drop to 3 sensors, below $K = 4$, and the error jumps to **0.13**.

**The argument is just counting.** A bandlimited graph signal lives in the span of the first $K$ harmonics, so it has only $K$ unknown coefficients no matter how many nodes the graph has. Read $m \geq K$ nodes and you have $m$ equations in $K$ unknowns; `lstsq` solves for the coefficients and $U_{:,:K}c$ reconstructs everything. Students often expect a deeper mechanism — there isn't one, and that is the point. This is the subspace version of the [Compressed Sensing](./Compressed_Sensing.ipynb) logic, but *easier*: there the support was unknown and the search combinatorial; here the subspace is known and it is a linear solve.

**The 3-sensor failure is aliasing, graph edition.** Three equations cannot determine four unknowns. Infinitely many bandlimited signals pass through those three readings, `lstsq` returns the minimum-norm one, and it is wrong. That is not a numerical problem and no better solver fixes it — below the required sample count, distinct signals are genuinely indistinguishable, exactly as sub-Nyquist sampling makes distinct sinusoids indistinguishable in classical DSP. The information is not there.

**Now the phrase carrying the real engineering: "well-chosen".** The recovery needs the selected rows of $U_{:,:K}$ to be well-conditioned. Twelve sensors clustered in one corner of the network would see nearly identical values, making those rows nearly parallel — the system would be numerically hopeless even with $m \gg K$. **Sensor placement matters as much as sensor count**, which is precisely the array-geometry lesson from [Array Processing](./Array_Processing.ipynb) transplanted onto an irregular graph.

That turns "where should the sensors go?" into a well-posed optimisation: choose the node subset maximising the conditioning (or the determinant, or the smallest singular value) of $U_{\text{sel},:K}$. There is a real literature here — greedy determinant maximisation, leverage-score sampling, E-optimal design — and it is genuinely useful. If you have budget for 12 sensors on a 500-node water network, this is the question you are asking.

**One honest caveat about how easy this instance is.** `x_band = x_clean` was *constructed* from exactly the first four harmonics, so it is perfectly bandlimited and the residual is pure floating-point. Real signals are only approximately bandlimited: their out-of-band energy sets an error floor that no sampling scheme removes, so you would never see 2e-15 on measured data. The mechanism is real; the precision here is a property of the synthetic setup.

---
### 🕐 Session 4 of 4 — *Message Passing = Learned Graph Filters* (~40 min)
**Goal:** build a GCN from scratch; classify nodes; see it as Session 2 with trained coefficients.
**Builds on:** Session 3; [CNN workshop](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb).

---

## 5. GNNs, Demystified

💡 **Intuition.** A graph-convolution layer is: *average your neighbors (a fixed 1-hop low-pass $\hat{A} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$), then apply a learned linear map and a nonlinearity*. Stack $k$ layers ⇒ $k$-hop receptive field — precisely Session 2's polynomial filters with coefficients chosen by gradient descent. It's the [CNN story](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) (weight sharing + locality) generalized to irregular neighborhoods.

In [ ]:
# two-community node classification (a planted partition on top of geometry)

# YOUR CODE HERE


**What just happened.** Ten labelled nodes, eighty to classify. The GCN reaches **88.6%** on the unseen nodes; the identical network without the graph reaches **41.4%**.

**This is a controlled ablation, which is what makes it worth anything.** Same architecture, same two input features, same ten labels, same optimiser, same number of steps. The *only* difference is whether $\hat A$ appears in the forward pass. So the gap is attributable to the graph and to nothing else.

**But read the 41.4% correctly.** With two roughly balanced classes, chance is about 50% — so the MLP scored *below* chance. It is not merely uninformative, it is anti-correlated with the truth. That is not evidence of the graph being 47 points valuable; it is what overfitting ten examples looks like. One of the two input features is deliberately useless random noise, and with ten training points the MLP can fit that noise and generalise backwards. On a 70-node test set, landing a few points either side of chance is ordinary variance.

**The honest comparison is 88.6% against ~50%**, and the MLP's exact value below chance is noise rather than a measurement. Anyone quoting "GCN beats MLP by 47 points" from this cell is over-reading it.

**Why the graph is worth so much here.** The features barely distinguish the classes — one is pure noise, the other a noisy hint. What the GCN adds is *structure*: neighbouring nodes tend to share labels (homophily), so averaging over neighbours both propagates the ten known labels outward and denoises the weak feature. Ten labels classify eighty nodes because the edges carry information that the node features do not.

That condition is load-bearing and worth naming: it only works under **homophily**. On a graph where connected nodes tend to *differ* — a fraud network where fraudsters transact with non-fraudsters, say — this same averaging destroys the signal, and specialised heterophilous architectures exist for exactly that case.

**And the layer is Session 2 with learned coefficients.** Decompose the forward pass: `Ahat @ X` averages neighbours (a fixed 1-hop low-pass), `W1` applies a learned linear map, `relu` adds the nonlinearity. Stack two layers and the receptive field is 2 hops — which is precisely a **degree-2 polynomial graph filter**, with the coefficients found by gradient descent instead of by least squares. GNNs are not a new mathematical object; they are the polynomial filters you built an hour ago, trained rather than designed.

Seen that way, the [CNN](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) connection is immediate too. A CNN shares weights across spatial positions with local receptive fields; a GCN shares weights across nodes with graph-local receptive fields. Given Session 1's ring oracle, a CNN is a GCN whose graph happens to be a grid — the harder part on a general graph is only that neighbourhoods vary in size and have no canonical ordering, which is why you *average* (permutation-invariant) rather than apply an ordered mask.

**One consequence worth knowing.** Repeated neighbour-averaging is repeated low-pass filtering, and enough of it drives every node toward the $\lambda \approx 0$ eigenvector — which is constant. That is **oversmoothing**: stack too many layers and all representations converge to the same value. It is why most GCNs are 2–3 layers deep while CNNs run to 50+, and through Session 2's lens it is obvious rather than mysterious.

Ten labels classify eighty nodes because the graph *propagates* them — message passing is label smoothing through a learned low-pass. (Also visible here: stack too many layers and everything averages toward mush — *oversmoothing*, the graph version of over-aggressive low-pass filtering.)

## 6. Conclusion

The Laplacian gives every network a Fourier basis (reducing to the DFT on a ring — verified); filters are functions of $L$, made local by polynomials; bandlimited signals need only $K$ good sensors; and GNNs are those polynomial filters with learned coefficients. Classical DSP was the special case all along.

---
## Where next

- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) — the eigen-machinery, if it felt fast.
- [CNN workshop](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) — the regular-grid special case.
- [Statistical SP](./Statistical_Signal_Processing.ipynb) — stochastic graph signals are an open research door.